In [ ]:
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

from utils import (
    get_embeddings,
    create_faiss_index,
    search_hotels_by_query,
    generate_hotel_answer_faiss,
    create_qdrant_collection,
    upload_reviews_to_qdrant,
    generate_hotel_answer_qdrant,
)

## Way 1 — FAISS (in-memory similarity search)

Baseline approach: embed the Paris-only reviews with `nomic-embed-text-v1.5`, build an in-process `faiss.IndexFlatIP` index (cosine similarity via L2-normalized vectors), retrieve the top-k most similar reviews, and generate a cited answer from them with Claude.

No persistence, no metadata filtering beyond the pre-filtered `df_paris` slice, no server — everything lives in this Python process and is rebuilt from scratch each run.

In [ ]:
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")
df = pd.DataFrame(dataset['train'])

df_paris = df.loc[df.locality == 'Paris']
df_paris.drop_duplicates()
reviews = df_paris['review_text'].tolist()

model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
if torch.cuda.is_available():
    model = model.to('cuda')

review_embeddings = get_embeddings(reviews, model)
faiss_index = create_faiss_index(review_embeddings)

In [ ]:
query = "Hotel with a view of the Eiffel tower."
json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)
print(json_output)

In [ ]:
query = "Hotels with a view of the Eiffel Tower"
response, sources = generate_hotel_answer_faiss(query, model, faiss_index, df_paris)
print(response)

The generated response showcases several key advantages over plain search results. First, it provides contextual analysis by comparing different hotels and their specific view offerings, from the Pullman's "exceptional view" with "stunning balcony vistas" to Citadines' more modest kitchen-area views. Second, it synthesizes information from multiple data points—room types, guest reviews, location details, and amenities—into coherent paragraphs that tell a complete story about each property. Third, it maintains source attribution through numbered citations, allowing users to trace specific claims back to their origins while presenting information in a natural, conversational format that's far more digestible than raw search results

## Way 2 — Qdrant (vector database)

Same embedding model, but indexed into a Qdrant collection instead of FAISS: the full (unfiltered) dataset is embedded and upserted with `hotel_name`, `review_text`, and `locality` stored as payload, so retrieval can filter by city at query time (`search_qdrant_with_filter`) instead of needing a separate pre-filtered DataFrame per city.

Qdrant runs in `:memory:` mode here for the notebook, but the same client code points at a real Qdrant server (`docker run -p 6333:6333 qdrant/qdrant`) to get persistence, concurrent access, and incremental upserts — the production-service properties FAISS doesn't provide.

In [ ]:
client = QdrantClient(":memory:")
text_embeddings_size = 768
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")
df_all = pd.DataFrame(dataset['train'])
df_all.drop_duplicates()
reviews_df = df_all.dropna(subset=['review_text'])
reviews = reviews_df['review_text'].tolist()

review_embeddings = get_embeddings(reviews, model)

In [ ]:
collection_name = "hotel_reviews"
create_qdrant_collection(client, collection_name, text_embeddings_size)

In [ ]:
upload_reviews_to_qdrant(client, collection_name, reviews_df, review_embeddings)

In [ ]:
query = "Amazing hotel close to everything"
city_filter = "Istanbul"
response_paris, sources_paris = generate_hotel_answer_qdrant(query, model, client, city=city_filter)

print("\n--- Generated Answer (City Filter) ---")

print("\n--- Sources Used (City Filter) ---")
for i, result in enumerate(sources_paris):
    print(f"Source {i+1}: Hotel: {result.payload.get('hotel_name', 'N/A')}, Locality: {result.payload.get('locality', 'N/A')}, Score: {result.score:.4f}")

print("\n" + "="*50 + "\n")